In [1]:
import pandas as pd
import numpy as np
import os

train_path = "../input/rsna-2022-cervical-spine-fracture-detection/train.csv"
train_df = pd.read_csv(train_path)

target_cols = ["patient_overall"] + [f"C{i}" for i in range(1, 8)]
train_df[target_cols] = train_df[target_cols].astype(float)

c_cols = [f"C{i}" for i in range(1, 8)]


def compute_label_means(df, alpha):
    """Laplace‑smoothed frequencies for C1‑C7 and the true patient_overall."""
    n = len(df)
    # Smoothed mean for each cervical vertebra label
    c_means = (df[c_cols].sum() + alpha) / (n + 2 * alpha)
    # Use the actual patient_overall column (not the derived any‑fracture flag)
    overall_raw = df["patient_overall"].mean()
    overall_smoothed = (overall_raw * n + alpha) / (n + 2 * alpha)
    means = {col: c_means[col] for col in c_cols}
    means["patient_overall"] = overall_smoothed
    return pd.Series(means)


def independence_patient_overall(label_means):
    """Independence‑based estimate for the any‑fracture label."""
    return 1.0 - np.prod(1.0 - label_means[c_cols])


def calibrate(p, beta):
    """Beta‑calibration."""
    p_beta = p**beta
    q_beta = (1.0 - p) ** beta
    return p_beta / (p_beta + q_beta)


def weighted_log_loss(df, preds, weights):
    """Weighted log‑loss averaged over all rows."""
    eps = 1e-15
    loss = 0.0
    total_w = 0.0
    for col in target_cols:
        y = df[col].values
        p = np.clip(preds[col], eps, 1 - eps)
        w = weights.get(col, 1.0)
        loss += w * (-(y * np.log(p) + (1 - y) * np.log(1 - p))).sum()
        total_w += w * len(y)
    return loss / total_w


np.random.seed(42)
indices = np.random.permutation(len(train_df))
split = int(0.8 * len(train_df))
train_idx, val_idx = indices[:split], indices[split:]

train_subset = train_df.iloc[train_idx].reset_index(drop=True)
val_subset = train_df.iloc[val_idx].reset_index(drop=True)

alpha_candidates = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 2.0]
blend_candidates = [i / 10.0 for i in range(0, 11)]  # 0.0 … 1.0
# Expanded beta range (0.5 … 2.0) for finer calibration
beta_candidates = [0.5 + 0.05 * i for i in range(0, 31)]  # 0.5 … 2.0

label_weights = {"patient_overall": 2.0}
for i in range(1, 8):
    label_weights[f"C{i}"] = 1.0

best_cfg = {}
best_score = float("inf")

for alpha in alpha_candidates:
    label_means = compute_label_means(train_subset, alpha)
    indep_overall = independence_patient_overall(label_means)

    for blend in blend_candidates:
        base_preds = {}
        for col in target_cols:
            if col == "patient_overall":
                raw = label_means["patient_overall"]
                blended = blend * indep_overall + (1 - blend) * raw
                base_preds[col] = blended
            else:
                base_preds[col] = label_means[col]

        for beta in beta_candidates:
            cal_preds = {col: calibrate(base_preds[col], beta) for col in target_cols}
            loss = weighted_log_loss(val_subset, cal_preds, label_weights)
            if loss < best_score:
                best_score = loss
                best_cfg = {
                    "alpha": alpha,
                    "blend": blend,
                    "beta": beta,
                    "label_means": label_means,
                    "indep_overall": indep_overall,
                }

alpha = best_cfg["alpha"]
blend_weight = best_cfg["blend"]
calibration_beta = best_cfg["beta"]  # retained for reference
label_means = compute_label_means(train_df, alpha)
indep_patient_overall = independence_patient_overall(label_means)

print(
    f"Chosen hyper‑params → alpha={alpha}, blend={blend_weight}, beta={calibration_beta:.3f}"
)
print(f"Validation weighted log‑loss with these params = {best_score:.5f}")

base_preds_val = {}
for col in target_cols:
    if col == "patient_overall":
        raw = label_means["patient_overall"]
        blended = blend_weight * indep_patient_overall + (1 - blend_weight) * raw
        base_preds_val[col] = blended
    else:
        base_preds_val[col] = label_means[col]

per_label_beta = {}
for col in target_cols:
    y = val_subset[col].values
    best_beta = calibration_beta
    best_loss = float("inf")
    for beta in beta_candidates:
        p = calibrate(base_preds_val[col], beta)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        loss = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        loss *= label_weights[col]
        if loss < best_loss:
            best_loss = loss
            best_beta = beta
    per_label_beta[col] = best_beta

print("Per‑label beta calibration:", per_label_beta)




Chosen hyper‑params → alpha=0.0, blend=0.0, beta=1.000
Validation weighted log‑loss with these params = 0.41429
Per‑label beta calibration: {'patient_overall': 1.05, 'C1': 1.35, 'C2': 1.0, 'C3': 0.95, 'C4': 1.05, 'C5': 1.2000000000000002, 'C6': 1.0, 'C7': 0.7}


In [2]:
test_path = "../input/rsna-2022-cervical-spine-fracture-detection/test.csv"
test_df = pd.read_csv(test_path)


def get_prediction(row):
    col = row["prediction_type"]
    if col == "patient_overall":
        raw = label_means["patient_overall"]
        blended = blend_weight * indep_patient_overall + (1 - blend_weight) * raw
        base = blended
    else:
        base = label_means[col]
    beta = per_label_beta[col]
    calibrated = calibrate(base, beta)
    return calibrated


test_df["fractured"] = test_df.apply(get_prediction, axis=1)
test_df["fractured"] = np.clip(test_df["fractured"], 1e-4, 1 - 1e-4)




In [3]:
submission_path = "submission.csv"
test_df[["row_id", "fractured"]].to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to submission.csv
